# BhramGuard Web Model Training

Train a phishing classifier for webpage or HTML-like content from `backend/ml/datasets/webs.csv`.

This notebook starts with a classical baseline: TF-IDF text features plus lightweight structural webpage features. It saves model artifacts into `backend/ml/models`.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

DATASET_PATH = PROJECT_ROOT / "backend" / "ml" / "datasets" / "webs.csv"
MODEL_DIR = PROJECT_ROOT / "backend" / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("Models:", MODEL_DIR)

Project root: c:\Users\Bidur\Desktop\BhramGuard-
Dataset: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\datasets\webs.csv
Models: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\models


In [2]:
import re

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [3]:
SAMPLE_ROWS = 50000

try:
    df_web = pd.read_csv(DATASET_PATH, nrows=SAMPLE_ROWS, on_bad_lines="skip")
except OSError as exc:
    raise RuntimeError(
        "Could not read webs.csv. Because this is a phishing webpage dataset, "
        "Windows Defender may block direct reads. If that happens, allowlist the "
        "project dataset folder or regenerate a sanitized sample CSV."
    ) from exc

print("Shape:", df_web.shape)
print("Columns:", df_web.columns.tolist())
df_web.head()

Shape: (15756, 2)
Columns: ['text', 'label']


,text,label
0,"<!doctypehtml PUBLIC ""-//W3C//DTD HTML 4.01 Tr...",1
1,<!doctypehtml><title>\n Segment - Math Open ...,0
2,<!doctypehtml><html itemscope itemtype=http://...,0
3,"<!doctypehtml PUBLIC ""-//W3C//DTD XHTML 1.0 Tr...",0
4,<title>\n Not Acceptable!\n </title><body><...,0


In [4]:
def infer_columns(df):
    label_candidates = ["label", "Label", "class", "target", "is_phishing"]
    text_candidates = ["text", "html", "content", "body", "web", "page"]

    label_col = next((col for col in label_candidates if col in df.columns), None)
    if label_col is None:
        raise ValueError(f"No label column found. Columns: {df.columns.tolist()}")

    text_col = next((col for col in text_candidates if col in df.columns and col != label_col), None)
    if text_col is None:
        object_cols = [col for col in df.columns if col != label_col and df[col].dtype == "object"]
        if not object_cols:
            raise ValueError(f"No text/html column found. Columns: {df.columns.tolist()}")
        text_col = object_cols[0]

    return text_col, label_col

TEXT_COL, LABEL_COL = infer_columns(df_web)
print("Text column:", TEXT_COL)
print("Label column:", LABEL_COL)
print(df_web[LABEL_COL].value_counts(dropna=False))

Text column: text
Label column: label
label
0    10264
1     5492
Name: count, dtype: int64


In [5]:
df_web = df_web[[TEXT_COL, LABEL_COL]].dropna()
df_web[TEXT_COL] = df_web[TEXT_COL].astype(str)
df_web[LABEL_COL] = df_web[LABEL_COL].astype(int)
df_web = df_web.drop_duplicates(subset=[TEXT_COL])

print("Cleaned shape:", df_web.shape)
print(df_web[LABEL_COL].value_counts(normalize=True))

Cleaned shape: (15701, 2)
label
0    0.653653
1    0.346347
Name: proportion, dtype: float64


In [6]:
def extract_web_features(value):
    text = "" if pd.isna(value) else str(value)
    lowered = text.lower()

    return {
        "char_length": len(text),
        "word_count": len(re.findall(r"\w+", text)),
        "link_count": len(re.findall(r"https?://|www\.", lowered)),
        "form_count": lowered.count("<form"),
        "password_count": lowered.count("password"),
        "input_count": lowered.count("<input"),
        "script_count": lowered.count("<script"),
        "iframe_count": lowered.count("<iframe"),
        "login_count": lowered.count("login"),
        "verify_count": lowered.count("verify"),
        "account_count": lowered.count("account"),
        "secure_count": lowered.count("secure"),
    }

manual_features = pd.DataFrame(df_web[TEXT_COL].apply(extract_web_features).tolist())
manual_features.describe().T

,count,mean,std,min,25%,50%,75%,max
char_length,15701.0,28271.400293,24068.752374,6.0,6955.0,22811.0,45338.0,101542.0
word_count,15701.0,4187.235144,3703.423413,1.0,923.0,3273.0,6793.0,25758.0
link_count,15701.0,68.000382,95.711869,0.0,5.0,30.0,90.0,890.0
form_count,15701.0,1.158716,1.644882,0.0,0.0,1.0,1.0,36.0
password_count,15701.0,3.677091,12.918591,0.0,0.0,0.0,3.0,705.0
input_count,15701.0,5.514999,9.287797,0.0,0.0,3.0,7.0,231.0
script_count,15701.0,12.151965,12.870535,0.0,3.0,9.0,17.0,172.0
iframe_count,15701.0,0.350360,1.099208,0.0,0.0,0.0,0.0,28.0
login_count,15701.0,6.756194,17.371917,0.0,0.0,1.0,5.0,531.0
verify_count,15701.0,0.212279,1.075242,0.0,0.0,0.0,0.0,50.0


In [7]:
X_text_train, X_text_test, X_manual_train, X_manual_test, y_train, y_test = train_test_split(
    df_web[TEXT_COL],
    manual_features,
    df_web[LABEL_COL],
    test_size=0.2,
    random_state=42,
    stratify=df_web[LABEL_COL],
)

tfidf = TfidfVectorizer(
    max_features=3000,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    stop_words="english",
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

X_train = hstack([X_train_tfidf, csr_matrix(X_manual_train.values)])
X_test = hstack([X_test_tfidf, csr_matrix(X_manual_test.values)])

print("Train matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

Train matrix: (12560, 3012)
Test matrix: (3141, 3012)


In [8]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1),
}

results = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    auc = roc_auc_score(y_test, probabilities) if probabilities is not None else None

    print("\n===", model_name, "===")
    print(classification_report(y_test, predictions))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))
    if auc is not None:
        print("ROC-AUC:", auc)

    results[model_name] = {"model": model, "auc": auc}

best_model_name = max(results, key=lambda name: results[name]["auc"] or 0)
best_model = results[best_model_name]["model"]
print("Best model:", best_model_name)

c:\Users\Bidur\Desktop\BhramGuard-\backend\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



=== logistic_regression ===
              precision    recall  f1-score   support

           0       0.87      0.69      0.77      2053
           1       0.58      0.81      0.68      1088

    accuracy                           0.73      3141
   macro avg       0.73      0.75      0.72      3141
weighted avg       0.77      0.73      0.74      3141

Confusion matrix:
[[1409  644]
 [ 203  885]]
ROC-AUC: 0.8235056839345578

=== random_forest ===
              precision    recall  f1-score   support

           0       0.94      0.97      0.96      2053
           1       0.94      0.89      0.91      1088

    accuracy                           0.94      3141
   macro avg       0.94      0.93      0.94      3141
weighted avg       0.94      0.94      0.94      3141

Confusion matrix:
[[1989   64]
 [ 118  970]]
ROC-AUC: 0.985323889358471
Best model: random_forest


In [9]:
artifact = {
    "model": best_model,
    "vectorizer": tfidf,
    "manual_feature_columns": manual_features.columns.tolist(),
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "best_model_name": best_model_name,
}

joblib.dump(artifact, MODEL_DIR / "web_phishing_model.pkl")
print("Saved:", MODEL_DIR / "web_phishing_model.pkl")

Saved: c:\Users\Bidur\Desktop\BhramGuard-\backend\ml\models\web_phishing_model.pkl
